# Continuous bandwidth minimizers: ISE, Q₀, Q₁, and Q₁₃/₁₄

For each $n=100,200,\ldots,2000$, compare the sampling distributions of four continuously optimized bandwidths on the common closed interval $h\in[0.01,2]$:

| Selector | Sample criterion |
|---|---|
| ISE | $\int(\widehat f_h-f)^2$ against the known $f=N(0,1)$ |
| $\widehat Q_0$ | $\tfrac12h^2\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_1$ | $\tfrac12\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_{13/14}$ | $\tfrac12h^{1/7}\|\partial_h\widehat f_h\|_2^2$ |

These subscripts use the thesis's **r parameter**, with
$\widehat Q_r(h)=\tfrac12h^{2(1-r)}\|\partial_h\widehat f_h\|_2^2$.
The ISE selector is the sample-specific oracle available here because the simulation density is known. Every criterion is evaluated with analytic Gaussian formulas. The search refines interior candidates continuously and compares both endpoints. Endpoint selections are part of the reported distributions.

The default is **M=1000 paired repetitions at each sample size**. The original 1000-by-1000 sample bank from seed `20260830` is retained exactly. A second block of 1000 observations per repetition is generated independently with `SeedSequence([20260830, 1])` and appended. Thus every sample for n≤1000 is unchanged, and the larger samples extend those same repetitions. Pre-unification results have been removed; new searches use the shared runner. A preview uses the first 12 repetitions of the same complete bank.

Sources: [thesis ISE and `prop:main_thm` defining Q_r](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [exact Gaussian ISE helper](../kde_tools/density.py); [continuous selector implementation and derived log-h derivatives](../kde_tools/continuous_minimizers.py); [earlier paired-sample settings](../inputs/qhat_sampling.json).

Run all cells. A separate resumable checkpoint is saved for each n. The figures are saved automatically as PDFs and also as PNGs. Two pages each show the complete interval and the interior selections with separate bars for the probability at the upper endpoint h=2. All histogram frequencies use M as denominator.

Reusable functions: [`continuous_experiments.py`](../kde_tools/continuous_experiments.py).
Experiment settings and calls are below. Every displayed figure is saved automatically as a PDF.


In [ ]:
from _setup import setup_notebook
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from kde_tools.continuous_experiments import (
    draw_histograms,
    run_sample_size,
    save_continuous_distributions,
    style_panel,
    validate_continuous_distributions,
)
from kde_tools.continuous_minimizers import CONTINUOUS_SELECTOR_VERSION, METHODS, R_VALUES
from kde_tools.continuous_runs import CONTINUOUS_RUNNER_VERSION
from kde_tools.energy_minimization import (
    DEFAULT_ENERGY_SEARCH,
    EnergySearchConfig,
    energy_search_settings,
)
from kde_tools.experiment_inputs import REFERENCE_SAMPLE_PATH, load_reference_sample
from kde_tools.qhat import GaussianQhatObjective, minimize_qhat
from kde_tools.thesis_exports import save_minimizer_panel
from matplotlib.lines import Line2D
from threadpoolctl import threadpool_limits

paths = setup_notebook("continuous_ise_qhat_distributions_vs_n")
code_directory = paths.code
pdf_output_directory = paths.pdfs
notebook_name = "continuous_ise_qhat_distributions_vs_n"
data_output_directory = paths.data
image_output_directory = paths.images


## Settings and paired Gaussian samples

The optimization interval is the same [0.01,2] used in the earlier experiment. `HISTOGRAM_POINTS` controls the common display bins only. The minimizers are continuously refined and can fall between bin centers. The three Q criteria are sample energies, optimized separately within each repetition.


In [ ]:
RUN_PROFILE = "thesis"
PROFILES = {"thesis": {"repetitions": 1000}, "preview": {"repetitions": 12}}
SAMPLE_SIZES = [100, 1000, 1500, 2000]
SEED = 20260830
SAMPLE_BANK_REPETITIONS = 1000
SAMPLE_BANK_MAXIMUM_N = 2000
SAMPLE_BANK_BLOCK_SIZE = 1000
H_MIN, H_MAX = (0.01, 2.0)
SIGMA2 = 1.0
SCAN_POINTS = DEFAULT_ENERGY_SEARCH.scan_points
SPATIAL_POINTS = DEFAULT_ENERGY_SEARCH.spatial_points
HALF_WIDTH = DEFAULT_ENERGY_SEARCH.half_width
HISTOGRAM_POINTS = 201
WORKERS = 6
REUSE_CHECKPOINTS = True
M = PROFILES[RUN_PROFILE]["repetitions"]
if not 1 <= M <= SAMPLE_BANK_REPETITIONS:
    raise ValueError("Repetitions must fit in the fixed paired sample bank.")
if not SAMPLE_SIZES or min(SAMPLE_SIZES) < 1 or max(SAMPLE_SIZES) > SAMPLE_BANK_MAXIMUM_N:
    raise ValueError("Sample sizes must fit in the fixed sample bank.")
sample_blocks = []
for block_index in range(
    (SAMPLE_BANK_MAXIMUM_N + SAMPLE_BANK_BLOCK_SIZE - 1) // SAMPLE_BANK_BLOCK_SIZE
):
    block_seed = SEED if block_index == 0 else np.random.SeedSequence([SEED, block_index])
    sample_blocks.append(
        np.random.default_rng(block_seed).normal(
            0.0, np.sqrt(SIGMA2), size=(SAMPLE_BANK_REPETITIONS, SAMPLE_BANK_BLOCK_SIZE)
        )
    )
full_sample_bank = np.concatenate(sample_blocks, axis=1)[:, :SAMPLE_BANK_MAXIMUM_N]
prefix_sample_bank_sha256 = hashlib.sha256(
    full_sample_bank[:, :SAMPLE_BANK_BLOCK_SIZE].tobytes()
).hexdigest()
full_samples = full_sample_bank[:M]
configuration = {
    "version": CONTINUOUS_SELECTOR_VERSION,
    "energy_search": energy_search_settings(),
    "runner": CONTINUOUS_RUNNER_VERSION,
    "methods": list(METHODS),
    "r_values": R_VALUES,
    "sample_sizes": SAMPLE_SIZES,
    "repetitions": M,
    "seed": SEED,
    "sigma2": SIGMA2,
    "h_min": H_MIN,
    "h_max": H_MAX,
    "sample_bank_shape": list(full_sample_bank.shape),
    "sample_bank_generation": "fixed_1000_column_blocks_preserving_original_prefix_v1",
    "sample_bank_block_size": SAMPLE_BANK_BLOCK_SIZE,
    "prefix_sample_bank_sha256": prefix_sample_bank_sha256,
    "sample_bank_sha256": hashlib.sha256(full_sample_bank.tobytes()).hexdigest(),
    "scan_points": SCAN_POINTS,
    "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH,
    "histogram_points": HISTOGRAM_POINTS,
    "source_sha256": {
        filename: hashlib.sha256((code_directory / "kde_tools" / filename).read_bytes()).hexdigest()
        for filename in ("continuous_minimizers.py", "qhat.py")
    },
}
configuration_json = json.dumps(configuration, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(json.dumps(configuration, indent=2) + "\n")
histogram_bandwidths = np.geomspace(H_MIN, H_MAX, HISTOGRAM_POINTS)
histogram_positions = np.arange(HISTOGRAM_POINTS, dtype=float)
histogram_edges = np.arange(HISTOGRAM_POINTS + 1) - 0.5
continuous_cache_root = code_directory / "outputs/data/shared_continuous_minimizers"
continuous_import_directories = []
print(f"M={M} paired repetitions; n={SAMPLE_SIZES}; continuous interval=[{H_MIN},{H_MAX}]")
print(f"Checkpoint and CSV directory: {run_directory}")


## Compute or resume all four continuous minimizers

The same sample supplies all four selectors. A Fourier scan supplies candidate basins, followed by analytic Gaussian-pair refinement and comparison with both endpoints. A finite scan is a numerical search, rather than a certificate that every stationary point was enumerated. Later validation doubles both scan and Fourier resolution on three repetitions at every n.


In [ ]:
search_settings = {
    "h_min": H_MIN,
    "h_max": H_MAX,
    "sigma2": SIGMA2,
    "scan_points": SCAN_POINTS,
    "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH,
}
results_by_n = {}
with threadpool_limits(limits=1):
    for current_n in SAMPLE_SIZES:
        results_by_n[current_n] = run_sample_size(
            current_n,
            reuse=REUSE_CHECKPOINTS,
            workers=WORKERS,
            configuration=configuration,
            continuous_cache_root=continuous_cache_root,
            continuous_import_directories=continuous_import_directories,
            full_samples=full_samples,
            run_directory=run_directory,
            search_settings=search_settings,
        )


## Saved distributions, boundary counts, and summaries

Each CSV selection row records a continuously optimized bandwidth and its numerical diagnostics. Histogram frequencies use the same geometric bin centers for all four selectors and divide by M. The summary explicitly counts selections at both endpoints. Interior-only frequencies below keep the same denominator M.

`peak_comparison.csv` records the ISE mode bin, the green interior mode bin, boundary mass, and paired bandwidth comparisons on samples where Q₁₃/₁₄ has an interior minimizer. These are descriptive histogram and sample comparisons.


In [ ]:
frequencies, boundary_masses = save_continuous_distributions(
    histogram_points=HISTOGRAM_POINTS,
    repetitions=M,
    sample_sizes=SAMPLE_SIZES,
    histogram_bandwidths=histogram_bandwidths,
    histogram_edges=histogram_edges,
    histogram_positions=histogram_positions,
    results_by_n=results_by_n,
    run_directory=run_directory,
)


## Numerical validation

Repeat the continuous searches at doubled scan and Fourier resolution on three paired repetitions at each n. Independently evaluate the exact ISE helper at all four selected bandwidths and both endpoints for these samples; the selected ISE must be no larger than any of those alternatives. Check stationarity and positive curvature for all interior solutions, plus the appropriate one-sided derivative signs at endpoints.


In [ ]:
validation_report = validate_continuous_distributions(
    half_width=HALF_WIDTH,
    h_max=H_MAX,
    h_min=H_MIN,
    repetitions=M,
    sample_sizes=SAMPLE_SIZES,
    scan_points=SCAN_POINTS,
    sigma2=SIGMA2,
    spatial_points=SPATIAL_POINTS,
    full_samples=full_samples,
    results_by_n=results_by_n,
    run_directory=run_directory,
)


## Distribution plots

The first figure shows all selections on the complete bandwidth interval. In the second, each main panel shows the histograms after removing only exact upper-endpoint selections; the inset bars show the probability at h=2. Main-panel frequencies still divide by all M repetitions. The main-panel range includes every occupied interior bin across all sample sizes and methods. Together, the interior histograms and inset bars account for the complete probability distributions.

The twenty sample sizes are split into two readable pages: n=100,…,1000 and n=1100,…,2000. Axis limits are shared across both pages.


In [ ]:
COLORS = {"ISE": "#7B4AB2", "Q0": "#0072B2", "Q1": "#D55E00", "Q13_14": "#009E73"}
LABELS = {
    "ISE": "ISE",
    "Q0": "$\\widehat Q_0$",
    "Q1": "$\\widehat Q_1$",
    "Q13_14": "$\\widehat Q_{13/14}$",
}
STYLES = {"ISE": "-", "Q0": "-", "Q1": "--", "Q13_14": "-."}
legend_handles = [
    Line2D([0], [0], color=COLORS[m], linestyle=STYLES[m], linewidth=2, label=LABELS[m])
    for m in METHODS
]
full_y_max = min(
    1.06, 1.06 * max((v.max() for point in frequencies.values() for v in point.values()))
)
full_ticks = np.unique(np.linspace(0, HISTOGRAM_POINTS - 1, 7, dtype=int))
for current_n in SAMPLE_SIZES:
    panel, panel_ax = plt.subplots(figsize=(4.2, 3.0), layout="constrained")
    draw_histograms(
        panel_ax,
        frequencies[current_n],
        colors=COLORS,
        styles=STYLES,
        histogram_edges=histogram_edges,
    )
    style_panel(
        panel_ax,
        current_n,
        full_ticks,
        full_y_max,
        HISTOGRAM_POINTS - 0.3,
        histogram_bandwidths=histogram_bandwidths,
    )
    boundary_ax = panel_ax.inset_axes([0.56, 0.48, 0.28, 0.42])
    boundary_ax.bar(
        np.arange(len(METHODS)),
        [boundary_masses[current_n][m] for m in METHODS],
        color=[COLORS[m] for m in METHODS],
        width=0.67,
    )
    boundary_ax.set_ylim(0, 1.05)
    boundary_ax.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
    boundary_ax.set_xticks([])
    boundary_ax.tick_params(axis="y", labelsize=8, length=2)
    boundary_ax.set_title(f"at $h={H_MAX:g}$", fontsize=9, pad=4)
    boundary_ax.patch.set_facecolor("#FAFBFD")
    for side in ("top", "right"):
        boundary_ax.spines[side].set_visible(False)
    panel_ax.set_title(f"$n={current_n}$; $M={M}$", fontsize=12, pad=8)
    panel.legend(
        handles=legend_handles, loc="outside lower center", ncol=4, frameon=False, fontsize=8
    )
    save_minimizer_panel(panel, current_n)
    plt.show()
    plt.close(panel)


## Local minima and interval endpoints: discussion control

This fixed-sample control supports the chapter's local-minimum discussion TODO.
It shows detected interior minima and both endpoint comparisons together;
it does not add a second Monte Carlo distribution study.

Sources: [shared search](../kde_tools/energy_minimization.py),
[Gaussian energy](../kde_tools/qhat.py), and [recorded input](../inputs/README.md).


In [ ]:
KDE_DIR = code_directory
NOTEBOOK_NAME = "continuous_ise_qhat_distributions_vs_n"
pdf_directory = pdf_output_directory


In [ ]:
SEARCH = EnergySearchConfig()
H_MIN, H_MAX = (0.01, 2.0)
R_VALUES = (0.0, 13 / 14, 1.0)
sample_path = REFERENCE_SAMPLE_PATH
samples, _ = load_reference_sample()
objective = GaussianQhatObjective(samples)
analyses = {str(r): minimize_qhat(objective, H_MIN, H_MAX, r=r, config=SEARCH) for r in R_VALUES}
summary = pd.DataFrame(
    [
        dict(
            r=r,
            local_minima=len(analyses[str(r)]["local_minima"]),
            interval_minimum_h=analyses[str(r)]["global_minimum"]["h"],
            interval_minimum_energy=analyses[str(r)]["global_minimum"]["energy"],
            boundary=analyses[str(r)]["global_minimum"]["boundary"],
            resolution_stable=analyses[str(r)]["diagnostics"]["resolution_stable"],
        )
        for r in R_VALUES
    ]
)
display(summary)


In [ ]:
local_minima = pd.DataFrame(
    [dict(r=r, **point) for r in R_VALUES for point in analyses[str(r)]["local_minima"]]
)
display(local_minima)
print("Search settings:", energy_search_settings(SEARCH))
settings = dict(
    search=energy_search_settings(SEARCH),
    h_min=H_MIN,
    h_max=H_MAX,
    r_values=R_VALUES,
    samples_sha256=hashlib.sha256(samples.tobytes()).hexdigest(),
)
run_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
output_directory = KDE_DIR / "outputs/data" / NOTEBOOK_NAME / "local_minimum_control" / run_id
output_directory.mkdir(parents=True, exist_ok=True)
(output_directory / "analysis.json").write_text(
    json.dumps(dict(settings=settings, analyses=analyses), indent=2) + "\n"
)
summary.to_csv(output_directory / "summary.csv", index=False)
local_minima.to_csv(output_directory / "local_minima.csv", index=False)


In [ ]:
h = np.geomspace(H_MIN, H_MAX, 601)
fig, axes = plt.subplots(1, len(R_VALUES), figsize=(13, 4.2), constrained_layout=True)
for ax, r in zip(np.atleast_1d(axes), R_VALUES):
    report = analyses[str(r)]
    ax.loglog(h, [objective.log_energy(np.log(x), r)[0] for x in h], color="#24658a")
    for i, point in enumerate(report["local_minima"]):
        ax.scatter(
            point["h"],
            point["energy"],
            color="#d88321",
            s=40,
            label="Interior local minima" if i == 0 else None,
            zorder=3,
        )
    best = report["global_minimum"]
    ax.scatter(
        best["h"],
        best["energy"],
        color="#744da9",
        marker="*",
        s=115,
        label="Interval minimum",
        zorder=4,
    )
    ax.set(xlabel="Bandwidth h", ylabel="Energy", title=f"r = {r:.6g}")
    ax.legend(fontsize=8)
fig.savefig(output_directory / "energy_minima.png", dpi=160)
plt.show()
plt.close(fig)
print("Full reports:", output_directory / "analysis.json")
print("Automatic PDF exports:", pdf_directory)
